# Ivy's Sentiment Analysis Project
## SVM and Bidirectional LSTM (Bi-LSTM)

In this project, I am comparing two very different types of models:
1. **Support Vector Machine (SVM):** A traditional machine learning model that looks for a 'boundary' between different sentiments.
2. **Bidirectional LSTM (Bi-LSTM):** A deep learning model that reads sentences both forward and backward to understand the full context.

I'll be training them on social media data and then testing them on real messages from Gmail and WhatsApp to see if they can understand student life!

### Step 1: Getting the Tools Ready
I'm importing pandas for data, sklearn for SVM, and TensorFlow for the Bi-LSTM model.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import re
import string
import warnings
warnings.filterwarnings('ignore')

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score
from sklearn.preprocessing import LabelEncoder

import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Bidirectional, Dense, Dropout, SpatialDropout1D

print("All libraries loaded successfully!")

### Step 2: Loading the Data
I'll load the training data and our special test set that contains the real Gmail and WhatsApp messages.

In [ ]:
# Loading from the project folders
train_df = pd.read_csv('../data/processed/processed_training_dataset.csv')
val_df = pd.read_csv('../data/processed/processed_validation_datset.csv')
test_df = pd.read_csv('../data/processed/student_test_dataset.csv')
print(f"Training set size: {len(train_df)}")
print(f"Student test set size: {len(test_df)}")

train_df.head(3)

## SECTION 1: EDA (Ivy - SVM & Bi-LSTM)

### Step 3: Visualizing Sentiment Distribution


In [ ]:
plt.figure(figsize=(8, 5))
sns.countplot(x='sentiment', data=train_df, palette='viridis')
plt.title('How many messages for each feeling?')
plt.show()

# SECTION 2: PREPROCESSING
### Step 4: Surgical Cleaning the text
Student communications (especially Gmail and WhatsApp) are full of technical noise like 'MTN:', 'GitHub:', and automated footers. I will use a 'Surgical Cleaning' approach to remove this noise and map student slang to proper English. This helps the model focus on the actual sentiment.

In [ ]:
# Improved slang mapping including WhatsApp shortcuts
slang_dictionary = {
    r'\bu\b': 'you', r'\bpls\b': 'please', r'\bplz\b': 'please',
    r'\btmrw\b': 'tomorrow', r'\bwat\b': 'what', r'\bwud\b': 'would',
    r'\bcuz\b': 'because', r'\bbtw\b': 'by the way', r'\bidk\b': 'i do not know',
    r'\bomg\b': 'oh my god', r'\blol\b': 'laughing', r'\bthx\b': 'thanks',
    r'\bsry\b': 'sorry', r'\bwanna\b': 'want to', r'\bgonna\b': 'going to',
    r'\bur\b': 'your', r'\br\b': 'are', r'\bn\b': 'and', r'\bok\b': 'okay'
}

def surgical_cleaner(text):
    if type(text) != str: return ""
    
    # 1. Basic normalization
    text = text.lower()
    
    # 2. Remove Technical Noise (Critical for Gmail/WhatsApp)
    text = re.sub(r'http\S+|www\.\S+|@\w+', '', text)
    text = re.sub(r'<.*?>', '', text) # HTML tags
    
    # Extended service prefix list including variations found in test data
    service_prefixes = (
        r'^(mtn|airtel|github|slack|gmail|whatsapp|udemy|vultr|codemagic|linkedin|' 
        r'amazon|railway|netlify|heroku|paystack|digitalocean|vercel|coursera|' 
        r'service termination notice|billing alert|security alert|reminder|alert|' 
        r'forwarded message|from:|to:|subject:|date:|sent:|on behalf of):\s*'
    )
    text = re.sub(service_prefixes, '', text)
    
    # 3. Remove footers and automated signatures
    text = re.sub(r'sent from my (iphone|android|mobile)', '', text)
    text = re.sub(r'please consider the environment before printing', '', text)
    text = re.sub(r'--- forwarded message ---', '', text)
    
    # 4. Remove numeric suffixes (e.g., "...at 5 PM. 17") often found in exports
    text = re.sub(r'\s\d+$', '', text)
    
    # 5. Handle contractions
    text = re.sub(r"can't", "cannot", text)
    text = re.sub(r"don't", "do not", text)
    text = re.sub(r"isn't", "is not", text)
    
    # 6. Replace slang
    for slang, correct in slang_dictionary.items():
        text = re.sub(slang, correct, text)
        
    # 7. Remove punctuation and extra spaces
    text = text.translate(str.maketrans('', '', string.punctuation))
    text = re.sub(r'\s+', ' ', text).strip()
    
    return text if text else "notification"# Applying the surgical cleaner
train_df['clean'] = train_df['text'].apply(surgical_cleaner)
val_df['clean'] = val_df['text'].apply(surgical_cleaner)
test_df['clean'] = test_df['text'].apply(surgical_cleaner)

print("Surgical cleaning is done!")

### Step 5: Encoding Labels
I will map the sentiments to three clean categories: Negative (0), Neutral (1), and Positive (2). This is important because our test set has a lot of Neutral messages that we don't want to miss!

In [ ]:
# Mapping dictionary for 3-class sentiment
label_mapping = {"Negative": 0, "Neutral": 1, "Positive": 2}

train_df['label'] = train_df['sentiment'].map(label_mapping)
val_df['label'] = val_df['sentiment'].map(label_mapping)
test_df['label'] = test_df['sentiment'].map(label_mapping)

# Drop any rows that couldn't be mapped (like 'Irrelevant' labels)
train_df = train_df.dropna(subset=['label'])
val_df = val_df.dropna(subset=['label'])
test_df = test_df.dropna(subset=['label'])

y_train = train_df['label'].astype(int).values
y_val = val_df['label'].astype(int).values
y_test = test_df['label'].astype(int).values

print("Labels mapped successfully: Negative=0, Neutral=1, Positive=2")

## SECTION 3: MODEL TRANSFORMATIONS (Ivy - SVM & Bi-LSTM)

In [ ]:
# 1. TF-IDF Vectorization
tfidf = TfidfVectorizer(max_features=10000, ngram_range=(1,2))
X_train_tfidf = tfidf.fit_transform(train_df['clean'])
X_val_tfidf = tfidf.transform(val_df['clean'])
X_test_tfidf = tfidf.transform(test_df['clean'])

print("TF-IDF Vectorization complete!")

## SECTION 4: TRAINING AND EVALUATION (Ivy - SVM & Bi-LSTM)

### Step 5.1: Tuning SVM Hyperparameters
I will test different values for the 'C' parameter (regularization) to see which one works best on our validation set. This is like trying on different sizes of shoes to find the best fit!

In [ ]:
best_c = 1.0
best_f1 = 0

for c_val in [0.01, 0.1, 1.0, 10.0]:
    temp_svm = SVC(kernel='rbf', C=c_val, random_state=42)
    temp_svm.fit(X_train_tfidf, y_train)
    temp_preds = temp_svm.predict(X_val_tfidf)
    current_f1 = f1_score(y_val, temp_preds, average='macro')
    
    print(f"Testing SVM with C={c_val}, Validation F1: {current_f1:.4f}")
    
    if current_f1 > best_f1:
        best_f1 = current_f1
        best_c = c_val

print(f"\nSuccess! The best C value is {best_c} with an F1-score of {best_f1:.4f}")

### Step 6: Final Support Vector Machine (SVM) Training
I will use the best 'C' value I found above to train my final SVM model.

In [ ]:
# 2. Training the SVM model with the BEST parameters
svm_model = SVC(kernel='rbf', C=best_c, random_state=42)
svm_model.fit(X_train_tfidf, y_train)

# 3. Validation results
svm_val_preds = svm_model.predict(X_val_tfidf)
print("SVM Validation Accuracy:", accuracy_score(y_val, svm_val_preds))
print("\nClassification Report:\n", classification_report(y_val, svm_val_preds, target_names=label_mapping.keys()))

### Step 7: Bidirectional LSTM (Bi-LSTM)
Now for the deep learning part. I'll tokenize the sentences and pad them to a length of 100 words. This is a bit more complex than SVM but usually more powerful!

In [ ]:
MAX_NB_WORDS = 20000
MAX_LEN = 100

# 1. Tokenizing
tokenizer = Tokenizer(num_words=MAX_NB_WORDS, oov_token='<OOV>')
tokenizer.fit_on_texts(train_df['clean'])

X_train_seq = pad_sequences(tokenizer.texts_to_sequences(train_df['clean']), maxlen=MAX_LEN)
X_val_seq = pad_sequences(tokenizer.texts_to_sequences(val_df['clean']), maxlen=MAX_LEN)
X_test_seq = pad_sequences(tokenizer.texts_to_sequences(test_df['clean']), maxlen=MAX_LEN)

# 2. Defining the Bi-LSTM Model Architecture
model = Sequential([
    Embedding(MAX_NB_WORDS, 128, input_length=MAX_LEN),
    SpatialDropout1D(0.3),
    Bidirectional(LSTM(64)),
    Dropout(0.5),
    Dense(64, activation='relu'),
    Dense(3, activation='softmax')
])

model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
model.summary()

### Step 7.1: Tuning Bi-LSTM Parameters
Deep learning models have many knobs to turn. We will test two different learning rates to see which one makes our model learn better from the validation set.

In [ ]:
best_lr = 0.001
best_val_acc = 0

for lr in [0.001, 0.0001]:
    print(f"\n--- Testing Bi-LSTM with Learning Rate: {lr} ---")
    temp_model = Sequential([
        Embedding(MAX_NB_WORDS, 128, input_length=MAX_LEN),
        SpatialDropout1D(0.3),
        Bidirectional(LSTM(64)),
        Dropout(0.5),
        Dense(64, activation='relu'),
        Dense(3, activation='softmax')
    ])
    temp_model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=lr), 
                       loss='sparse_categorical_crossentropy', metrics=['accuracy'])
    
    temp_hist = temp_model.fit(X_train_seq, y_train, epochs=2, batch_size=64, 
                               validation_data=(X_val_seq, y_val), verbose=1)
    
    val_acc = max(temp_hist.history['val_accuracy'])
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        best_lr = lr

print(f"\nSuccess! The best Learning Rate is {best_lr} with Validation Accuracy of {best_val_acc:.4f}")

### Step 8: Training the Final Bi-LSTM
I'll train for 5 epochs with our best learning rate found in the experiment above.

In [ ]:
# Re-build the model with the BEST learning rate found
model = Sequential([
    Embedding(MAX_NB_WORDS, 128, input_length=MAX_LEN),
    SpatialDropout1D(0.3),
    Bidirectional(LSTM(64)),
    Dropout(0.5),
    Dense(64, activation='relu'),
    Dense(3, activation='softmax')
])
model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=best_lr), 
              loss='sparse_categorical_crossentropy', metrics=['accuracy'])

history = model.fit(
    X_train_seq, y_train, 
    epochs=5, 
    batch_size=64, 
    validation_data=(X_val_seq, y_val), 
    verbose=1
)

### Step 9: Testing on Real Student Data
Now I will test both models on the real Gmail and WhatsApp data. Let's see how they perform!

In [ ]:
# SVM Test Results
svm_test_preds = svm_model.predict(X_test_tfidf)
svm_test_acc = accuracy_score(y_test, svm_test_preds)

# Bi-LSTM Test Results
lstm_test_probs = model.predict(X_test_seq)
lstm_test_preds = np.argmax(lstm_test_probs, axis=1)
lstm_test_acc = accuracy_score(y_test, lstm_test_preds)

print(f"Final SVM Test Accuracy: {svm_test_acc*100:.2f}%")
print(f"Final Bi-LSTM Test Accuracy: {lstm_test_acc*100:.2f}%")

# Comparing both models visually
acc_scores = [svm_test_acc, lstm_test_acc]
model_names = ['SVM', 'Bi-LSTM']

plt.figure(figsize=(7, 5))
sns.barplot(x=model_names, y=acc_scores, palette='coolwarm')
plt.axhline(0.8, color='red', linestyle='--', label='80% Target')
plt.ylim(0, 1)
plt.title('Which model understood students better?')
plt.legend()
plt.show()

### Step 10: Confusion Matrix
Let's see where our best model (Bi-LSTM) made mistakes in the student data.

In [ ]:
cm = confusion_matrix(y_test, lstm_test_preds)
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=label_mapping.keys(), yticklabels=label_mapping.keys())
plt.xlabel('Predicted Label')
plt.ylabel('True Label')
plt.title('Bi-LSTM Mistakes on Student Data')
plt.show()